# RepairScope-X — Notebook 14
## ALoDLM-1.7B Full-Precision Runtime + Own-Failure Collection

Notebook 13 independently confirmed Dream self-repair on held-out problems. We now move to a **second genuine diffusion language model** rather than extending Dream further.

**Model:** `amazon/ALoDLM-1.7B`

This notebook uses Amazon's official portable ALoDLM decoder and attempts **FP32 CUDA inference** on the available Colab GPU. It does not quantize the model.

### Goal
First establish a clean second-model runtime and collect **ALoDLM's own natural failures**. If enough failures are obtained, the next notebook will test ALoDLM → ALoDLM suffix self-repair.

### Balanced cohort
A deterministic SHA-256 selection with seed `3600` embeds:

- 8 Countdown
- 8 GSM8K
- 8 SVAMP

Total: **24 native ALoDLM generations**.

### Frozen public implementation
- ALoDLM code commit: `fe225a99e0b5aeff351804e871d7581155d7f27d`
- portable decoder
- `mode="left1"`
- `q=0.4`
- window size 16
- max 96 new tokens
- FP32

The model snapshot SHA is resolved once at runtime and recorded in the manifest.

Embedded benchmark SHA-256: `4097b3a3b371f9821be9e21e9cf1a1866d8f618652576e25f3c52bda84c0ebaa`

In [ ]:
!rm -rf /content/ALoDLM
!git clone -q https://github.com/amazon-science/ALoDLM.git /content/ALoDLM
!cd /content/ALoDLM && git checkout -q fe225a99e0b5aeff351804e871d7581155d7f27d
!pip -q install "transformers==4.56.1" "accelerate==1.10.1" "huggingface-hub>=0.34,<1.0" "safetensors>=0.6" "PyYAML>=6"
!pip -q install -e /content/ALoDLM --no-deps

## 1. Runtime preflight

In [ ]:
import os, gc, json, math, re, time, random, hashlib, platform, ast as pyast, subprocess, sys
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import torch, transformers, accelerate, huggingface_hub

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

BASE_SEED = 3600
random.seed(BASE_SEED)
np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(BASE_SEED)
    torch.cuda.empty_cache()

gc.collect()

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("CUDA:", torch.cuda.is_available())

assert torch.cuda.is_available(), "Notebook 14 requires a CUDA GPU runtime."

GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
print("GPU:", GPU_NAME, "| VRAM:", round(GPU_GB, 2), "GB")
print("Allocated:", round(torch.cuda.memory_allocated()/2**30, 3), "GB")

if GPU_GB < 14:
    raise RuntimeError("FP32 ALoDLM-1.7B expects roughly T4-class (~15 GB) VRAM or larger.")

if torch.cuda.memory_allocated()/2**30 > 1.0:
    raise RuntimeError("GPU memory is already occupied. Restart runtime and run from the top.")

torch.set_float32_matmul_precision("high")
print("Runtime preflight: PASS")

## 2. Verify pinned ALoDLM implementation

In [ ]:
REPO_DIR = Path("/content/ALoDLM")
EXPECTED_CODE_COMMIT = "fe225a99e0b5aeff351804e871d7581155d7f27d"

actual_commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()

assert actual_commit == EXPECTED_CODE_COMMIT
sys.path.insert(0, str(REPO_DIR))

from alodlm.generate import load_decoder
from alodlm.inference import DecodeConfig

print("ALoDLM commit:", actual_commit)
print("Pinned source validation: PASS")

## 3. Recover embedded benchmark

In [ ]:
import base64, zlib

EMBEDDED_PAYLOAD_SHA256 = "4097b3a3b371f9821be9e21e9cf1a1866d8f618652576e25f3c52bda84c0ebaa"
raw_payload = zlib.decompress(base64.b64decode('''eNq1WW2P27gR/ivE4oB88S4sWZJfULRI0Rwuh/ZQJC2Ctij2aImyCEuiSlL2+Q733/sMKdmyd5PVOvGHBRwPZzjzzMwzQ+e3OyNKkVqRPRrV6lSYu9V/frtrtFqXonqU2d3qLlVtbTO1r1cpL+VacytVvQoW03g1W9xN7jJuuRF2ePIx18IUkDVcW0nnSXrSJolWVUNK/zSC2UKwuq3WQhsWTViAP15nLFwy8QtPbXlgqk5xTDGoZW3qNSzXG2HZfRA8sA/CtrpmnO1wS8a4lraohJUpLDRwxuBWZ1Pgbw8h+/n79z+9/evj258+fnr3YcX+4O//488PcM5bvluR6envk/GAJN8ej2AKQBIPSBCMASSIlrcChEy/CpDwBhWyBCDzrkKmYwCJw+mtACHTrwIk/vZ4hBMW9h2zGIVHdDM4oleWx/QGcKA2wq5hwngMHuHiVnjA8qvwuEG3EJkShzg4ZqP442bs8TowgvkN0EgGaIxqltnNmmX2ymaZBd8ejxmwCDs8glFkGt2MS6PnqHRjqsV2ZTWX9WoeLJdnEDjhuMC/V5qJndAHtpZbwX4QNT5i+SnNhCE4aVjDEcp3C1YpTQHzevg9YQEXrEOCLDyw93lnBc4b9l389Awsqz2r2rRgmRKG7BGCvUG63aHktRZOx/zpbNxGL0CSJFF8LSQfcbdWrOCGGfkLs7KCj/hHxesDy3i7KSyVCIlVbVzAiCBttRY1FQkp2kIL4eRdsKSbFrLMcOgYdcF3YhhXGLwQVjyfz78m05SBnFcSXmrR1jgxYX8XCIcSum4Pst6ACliDFsoMUzlbq1rcy5pcT7fCF2/By5wKwTJeUa8xyAterVsEoYEH0v/GsI0iY0gnzLLQV8/JbqEsUNiYrqCeaPPSqHMThqqUl0hJJoy/fi9RKHshN0Xvk+hdGlxBJvsMnBzIlcq8ASrK9jBMw2z5UsPNguDaNLxlpm2ELgRKLEXsLZjHf9PCRWYaIVwX6LYm5kGuEC4QjvCxbq1APP8oOpUd/OcOfSBWs3f49/3f1BoqHqBMy52gVbm3Ax3g0uqHEyRekMNxuOQ8sr155yP5ISnxTtHny146QGrusiGM8cu8Nbu6mn9URc3WfEvhhSxVqQI4iCjlWrmuExz04oqDiihmyHxK4P3gWLsGuaRbh5r4X4uy8sRLlUHBHe145DOZ54K6m601Bgvu3eAGLXh2QCFplolSEoc+sE9kmipKS5Gjy0rBd65cBdtKm5KU6BDN5L5rCvQXGALdV0prS1JULhEfLeodQVivjLslGghRWnx0FQzN3t2TVyj2T5R56dWssrxrETuhWDwKk15RiwqJIICeIDjMZBS/xEtRFF7NS5rXqTRvemKiYg5Zu21LPMHhasQ2rYQnxk9jzlKhLRXdKWU7qUpJVPyOkt6pOkMRM1Yjvl7mTTlRciHyRi613ucDIE83dnKiG6IonDg5fOZu5xolJJr2aoOZ0MvdSHgKxcV8eCkNcXh1Q33iJc2BnIZvji992VHhFKwpuUWdVy5yVaH5U57nqsyocihKp9VwY9lf+I42n9zZ4mVJJ0BdNGOOo9Lfgb5B9/Esc44BXT8z/ZrhZq7nMNHAg7XI6fsCTSI7wDeayNyj7HqpcxJJwUl0Tw0EDdocfKpaP6W6RtiT4WNQe2Tcu63qIdrJSzvGcjb7iimQFkqVYM6csJGcBoHpgiKq94tmqtSWldzwTe04PsM31vINwpXdKACFgwYdyfcHKUJJnYDZEg6G3pl1IsjJE5ND1dmXVI+tcYqAWgd6GYvhy2dUZe17abgWDa4H1bps+qDOdr7pc+PE7HjVdPlIsQTcL5PznLgDI3PiihaU66oaxQIV+CFcacKjPQ0At5/6xf/PbbrFUSwvKP0SdebSocHnH4CPKIn8dU3TnMotitHrRlqF1THllcvus1ZQhb6etQA5tICEHxzYi3CJAa4aUJuzJLLPGKCXheyMUE8uYrJhTjP/6MiX7ThHqAX52Xo0X0RjMjGbzq/NhIPRtTE2oMHwRTP4JkgG31ZqJ/2KRFG8Yanmv6LxsZkc+iZ7g5xp1y+A8aBaN7pZEB3HvTdBlvecpjSEy+Nwdbee9iXiIf+d24U6XaLqo+mzLWgMVlEQXYvVj8rQ+psRy3K97rZF2ihSVdJv2Djvtp5SoRSw3PenHG+7zlywWuxpnzAPF3H2Z6krC2dC9BsgfSQuJwvDeBej4k2uDfffaC2OxyV5NItZ05rivm18NDPkHvtN4SHYHCqWghGRJ+VayHM8Kcbda/aovcZ8WMzRsDh9tOEC/dXfdwnMUdMdOqqQ8aOH6uxNEY7CZb68Fph+r71kKYWdYgPX1oJoDZ5hH0XLu5Lw6afh6PrcsVx3gs0CZtyKjDbQsjkZcj8RRSdhW5+J6Ynh4D4C9rmT7vXV3Ynm8TtD1HEVPeC8J+dDYDFuCsRX9xNIFvTc0AvfQYnVut9ykuRcVEgizJ55UBe0ST+wMHnWQnfMhVkr15RnJfUFnc60L7ZLO2dVNgacYJp8PTEneG6YrfBPkYYWAtEv0l7gtgHQjQaZdnL/a9oCC4GgB1ivdITh7KgWfaT9RVhQFaKEC8OQl8momC8e7K+I+V8gdeLXMHEvLP+73eWjkfa04YAhhSCmd3HWT5Y47E8dl6BT6L3EPQOER3Hz7P7/30n3f5Nw8xFNjnBnyXT6+/8Bol/ftw=='''))
assert hashlib.sha256(raw_payload).hexdigest() == EMBEDDED_PAYLOAD_SHA256

UPSTREAM = json.loads(raw_payload.decode("utf-8"))
SOURCES = UPSTREAM["selected_sources"]

assert len(SOURCES) == 24
assert len({x["problem_id"] for x in SOURCES}) == 24
assert Counter(x["dataset"] for x in SOURCES) == {
    "countdown_fresh": 8,
    "gsm8k": 8,
    "svamp": 8,
}

print("Payload SHA:", EMBEDDED_PAYLOAD_SHA256)
print("Dataset balance:", dict(Counter(x["dataset"] for x in SOURCES)))
print("Benchmark validation: PASS")

## 4. Task verifiers

In [ ]:
FINAL_RE = re.compile(
    r"(?im)^[ \t]*FINAL_ANSWER[ \t]*:[ \t]*"
    r"([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)"
    r"[ \t]*$"
)
NUM_RE = re.compile(r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?")

def close_num(a, b):
    return a is not None and math.isclose(float(a), float(b), abs_tol=1e-8, rel_tol=1e-8)

def strict_final(text):
    vals = FINAL_RE.findall(str(text))
    if not vals:
        return None
    try:
        x = float(vals[-1])
        return x if math.isfinite(x) else None
    except Exception:
        return None

def last_number(text):
    vals = NUM_RE.findall(str(text).replace(",", ""))
    if not vals:
        return None
    try:
        x = float(vals[-1])
        return x if math.isfinite(x) else None
    except Exception:
        return None

COUNTDOWN_PROMPT_RE = re.compile(
    r"Use the numbers\s+(.+?)\s+exactly once\s+to produce the target\s+"
    r"([-+]?(?:\d+(?:\.\d*)?|\.\d+))",
    flags=re.I,
)
ARITH_RUN_RE = re.compile(r"[0-9eE+\-*/().=\s]{3,}")
ALLOWED_BINOPS = (pyast.Add, pyast.Sub, pyast.Mult, pyast.Div)
ALLOWED_UNARY = (pyast.UAdd, pyast.USub)

def parse_countdown_prompt(prompt):
    m = COUNTDOWN_PROMPT_RE.search(str(prompt))
    if m is None:
        raise ValueError("Countdown prompt format not recognized.")
    return [float(x) for x in NUM_RE.findall(m.group(1))], float(m.group(2))

def safe_arith(expr):
    try:
        tree = pyast.parse(expr, mode="eval")
    except Exception:
        return None
    literals = []
    def walk(node):
        if isinstance(node, pyast.Expression):
            return walk(node.body)
        if isinstance(node, pyast.Constant) and isinstance(node.value, (int, float)):
            value = float(node.value)
            literals.append(value)
            return value
        if isinstance(node, pyast.BinOp) and isinstance(node.op, ALLOWED_BINOPS):
            a, b = walk(node.left), walk(node.right)
            if isinstance(node.op, pyast.Add): return a + b
            if isinstance(node.op, pyast.Sub): return a - b
            if isinstance(node.op, pyast.Mult): return a * b
            if isinstance(node.op, pyast.Div):
                if abs(b) < 1e-15: raise ZeroDivisionError
                return a / b
        if isinstance(node, pyast.UnaryOp) and isinstance(node.op, ALLOWED_UNARY):
            v = walk(node.operand)
            return -v if isinstance(node.op, pyast.USub) else v
        raise ValueError
    try:
        value = walk(tree)
        if not math.isfinite(value):
            return None
        return value, literals
    except Exception:
        return None

def same_multiset(a, b):
    if len(a) != len(b):
        return False
    aa, bb = sorted(map(float, a)), sorted(map(float, b))
    return all(close_num(x, y) for x, y in zip(aa, bb))

def countdown_task_success(text, prompt):
    nums, target = parse_countdown_prompt(prompt)
    cleaned = (
        str(text)
        .replace("×", "*")
        .replace("÷", "/")
        .replace("\\times", "*")
        .replace("\\div", "/")
    )
    candidates = []
    for m in ARITH_RUN_RE.finditer(cleaned):
        run = m.group(0).strip()
        if any(op in run for op in ["+", "-", "*", "/"]):
            candidates.extend(x.strip() for x in run.split("=") if x.strip())

    for expr in sorted(set(candidates), key=len, reverse=True):
        parsed = safe_arith(expr)
        if parsed is None:
            continue
        value, literals = parsed
        if same_multiset(literals, nums) and close_num(value, target):
            return True, expr
    return False, None

def verify_generation(text, source):
    strict = strict_final(text)
    numeric = last_number(text)
    strict_ok = close_num(strict, source["target"])
    numeric_ok = close_num(numeric, source["target"])

    if source["dataset"] == "countdown_fresh":
        task_ok, expr = countdown_task_success(text, source["prompt"])
    else:
        task_ok, expr = numeric_ok, None

    return {
        "strict_prediction": strict,
        "numeric_prediction": numeric,
        "target": float(source["target"]),
        "strict_success": bool(strict_ok),
        "numeric_success": bool(numeric_ok),
        "task_success": bool(task_ok),
        "valid_countdown_expression": expr,
    }

demo = "Use the numbers 19, 29, and 24 exactly once to produce the target 14."
assert countdown_task_success("19 + 24 - 29 = 14", demo)[0]
assert not countdown_task_success("29 - 19 = 10", demo)[0]
print("Verifier regression tests: PASS")

## 5. Freeze the current ALoDLM model snapshot

In [ ]:
from huggingface_hub import model_info, snapshot_download

MODEL_ID = "amazon/ALoDLM-1.7B"
MODEL_REVISION = model_info(MODEL_ID).sha

print("Resolved model revision:", MODEL_REVISION)

MODEL_DIR = snapshot_download(
    repo_id=MODEL_ID,
    revision=MODEL_REVISION,
)

print("Frozen snapshot:", MODEL_DIR)

## 6. Load ALoDLM-1.7B in FP32

In [ ]:
from types import SimpleNamespace

torch.cuda.empty_cache()
gc.collect()
torch.cuda.reset_peak_memory_stats()

args = SimpleNamespace(
    model=MODEL_DIR,
    model_config=None,
    device="cuda",
    precision="fp32",
)

t0 = time.time()

try:
    decoder = load_decoder(args)
except torch.cuda.OutOfMemoryError as exc:
    torch.cuda.empty_cache()
    gc.collect()
    raise RuntimeError(
        "ALoDLM-1.7B FP32 does not fit this GPU. "
        "Do not silently quantize this registered full-precision condition; "
        "rerun on a larger-memory GPU."
    ) from exc

load_seconds = time.time() - t0
tokenizer = decoder.tokenizer
model = decoder.model

param_dtype = next(model.parameters()).dtype
assert param_dtype == torch.float32

print("MODEL LOAD: PASS")
print("Precision:", param_dtype)
print("Load seconds:", round(load_seconds, 2))
print("Allocated GB:", round(torch.cuda.memory_allocated()/2**30, 3))
print("Peak GB:", round(torch.cuda.max_memory_allocated()/2**30, 3))

## 7. Native decoder configuration

In [ ]:
Q = 0.4
TAU = 0.4
MODE = "left1"
WINDOW_SIZE = 16
MAX_NEW_TOKENS = 96

GEN_CONFIG = DecodeConfig(
    q=Q,
    tau=TAU,
    mode=MODE,
    window_size=WINDOW_SIZE,
    max_new_tokens=MAX_NEW_TOKENS,
    position_penalty=None,
)
GEN_CONFIG.validate()

print({
    "q": Q,
    "tau": TAU,
    "mode": MODE,
    "window_size": WINDOW_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS,
})

## 8. Native prompt encoding + generation runner

In [ ]:
PROMPT_SUFFIX = (
    "\n\nSolve the problem carefully. "
    "For ordinary arithmetic word problems, end with `FINAL_ANSWER: <number>`. "
    "For Countdown, include a valid arithmetic expression using every supplied number exactly once."
)

def encode_source(source):
    ids = tokenizer.apply_chat_template(
        [{"role": "user", "content": source["prompt"] + PROMPT_SUFFIX}],
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    return list(ids)

def run_generation(source):
    prompt_ids = encode_source(source)
    torch.cuda.reset_peak_memory_stats()

    result = decoder.generate(
        prompt_ids,
        GEN_CONFIG,
    )

    text = str(result["text"]).strip()
    verification = verify_generation(text, source)

    return {
        "generation_id": source["problem_id"] + "|alodlm_fp32",
        "problem_id": source["problem_id"],
        "dataset": source["dataset"],
        "partition": source["partition"],
        "prompt": source["prompt"],
        "target": source["target"],
        "output": text,
        "verification": verification,
        "strict_success": bool(verification["strict_success"]),
        "numeric_success": bool(verification["numeric_success"]),
        "task_success": bool(verification["task_success"]),
        "prompt_tokens": int(result["prompt_tokens"]),
        "generated_tokens": int(result["generated_tokens"]),
        "wall_seconds": float(result["wall_seconds"]),
        "prefill_seconds": float(result["prefill_seconds"]),
        "tokens_per_second": float(result["tokens_per_second"]),
        "mean_exit_depth": result["mean_exit_depth"],
        "loop_token_passes": int(result["loop_token_passes"]),
        "loop_token_passes_per_output_token": result["loop_token_passes_per_output_token"],
        "outer_steps": int(result["outer_steps"]),
        "recurrent_passes": int(result["recurrent_passes"]),
        "stop_reason": str(result["stop_reason"]),
        "peak_vram_gb": float(torch.cuda.max_memory_allocated()/2**30),
    }

print("Generation runner: READY")

## 9. One real ALoDLM smoke test

In [ ]:
smoke = run_generation(SOURCES[0])

print(json.dumps(
    {k:v for k,v in smoke.items() if k not in {"prompt","output","verification"}},
    indent=2,
))
print("\nOUTPUT\n", smoke["output"])

assert smoke["generated_tokens"] > 0
assert smoke["outer_steps"] > 0
assert smoke["recurrent_passes"] > 0
assert math.isfinite(smoke["wall_seconds"])

print("\nREAL ALoDLM FP32 SMOKE: PASS")

## 10. Output directory + immutable signature

In [ ]:
SAVE_TO_DRIVE = True
CLEAR_OLD_RESULTS = False

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path(
        "/content/drive/MyDrive/RepairScopeX/Notebook14_ALoDLM_FP32_Own_Failures"
    )
else:
    OUT_DIR = Path(
        "/content/RepairScopeX_Notebook14_ALoDLM_FP32_Own_Failures"
    )

OUT_DIR.mkdir(parents=True, exist_ok=True)

signature_payload = {
    "protocol_version": "alodlm17b-own-failures-v1",
    "embedded_payload_sha256": EMBEDDED_PAYLOAD_SHA256,
    "code_commit": EXPECTED_CODE_COMMIT,
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "precision": "fp32",
    "q": Q,
    "tau": TAU,
    "mode": MODE,
    "window_size": WINDOW_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "problem_ids": [x["problem_id"] for x in SOURCES],
}

RUN_SIGNATURE = hashlib.sha256(
    json.dumps(signature_payload, sort_keys=True).encode()
).hexdigest()

GENERATIONS_PATH = OUT_DIR / "alodlm_natural_generations.jsonl"
MANIFEST_PATH = OUT_DIR / "run_manifest.json"

if CLEAR_OLD_RESULTS:
    for p in [GENERATIONS_PATH, MANIFEST_PATH]:
        if p.exists():
            p.unlink()

if MANIFEST_PATH.exists():
    old = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    if old.get("run_signature") != RUN_SIGNATURE:
        raise RuntimeError("Existing Notebook 14 results use a different configuration.")
else:
    MANIFEST_PATH.write_text(
        json.dumps(
            {
                "created_utc": datetime.now(timezone.utc).isoformat(),
                "run_signature": RUN_SIGNATURE,
                **signature_payload,
                "gpu": GPU_NAME,
                "gpu_total_gb": GPU_GB,
                "torch": torch.__version__,
                "transformers": transformers.__version__,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

print("Run signature:", RUN_SIGNATURE)
print("Output:", OUT_DIR)

## 11. Resume-safe 24-generation sweep

In [ ]:
def load_completed(path):
    done = {}
    if not path.exists():
        return done

    with open(path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError as exc:
                raise RuntimeError(f"Malformed JSONL at line {line_number}.") from exc

            if rec.get("run_signature") != RUN_SIGNATURE:
                raise RuntimeError("Found a trial from another Notebook 14 configuration.")

            gid = rec["generation_id"]
            if gid in done:
                raise RuntimeError(f"Duplicate generation_id: {gid}")
            done[gid] = rec

    return done

EXPECTED_IDS = {
    source["problem_id"] + "|alodlm_fp32"
    for source in SOURCES
}
assert len(EXPECTED_IDS) == 24

completed = load_completed(GENERATIONS_PATH)
print("Already completed:", len(completed), "/ 24")

with open(GENERATIONS_PATH, "a", encoding="utf-8") as f:
    for i, source in enumerate(SOURCES, start=1):
        gid = source["problem_id"] + "|alodlm_fp32"

        if gid in completed:
            print(f"[{i}/24] SKIP {gid}")
            continue

        try:
            rec = run_generation(source)
            rec["run_signature"] = RUN_SIGNATURE
            rec["status"] = "ok"
            rec["error"] = None

        except torch.cuda.OutOfMemoryError as exc:
            torch.cuda.empty_cache()
            gc.collect()
            rec = {
                "generation_id": gid,
                "problem_id": source["problem_id"],
                "dataset": source["dataset"],
                "status": "oom",
                "error": str(exc),
                "run_signature": RUN_SIGNATURE,
            }

        except Exception as exc:
            rec = {
                "generation_id": gid,
                "problem_id": source["problem_id"],
                "dataset": source["dataset"],
                "status": "error",
                "error": f"{type(exc).__name__}: {exc}",
                "run_signature": RUN_SIGNATURE,
            }

        f.write(json.dumps(rec, ensure_ascii=False))
        f.write("\n")
        f.flush()

        if rec["status"] == "ok":
            print(
                f"[{i}/24] {gid} | task={rec['task_success']} | "
                f"tokens={rec['generated_tokens']} | steps={rec['outer_steps']} | "
                f"passes={rec['recurrent_passes']} | depth={rec['mean_exit_depth']}"
            )
        else:
            print(f"[{i}/24] {gid} | {rec['status']} | {rec['error']}")

print("ALoDLM sweep finished.")

## 12. Validate matrix and summarize

In [ ]:
records = load_completed(GENERATIONS_PATH)

missing = EXPECTED_IDS - set(records)
extra = set(records) - EXPECTED_IDS

print("Expected:", len(EXPECTED_IDS))
print("Found:", len(records))
print("Missing:", len(missing))
print("Extra:", len(extra))

if missing:
    raise RuntimeError("Sweep incomplete. Rerun the sweep cell; completed items will be skipped.")
if extra:
    raise RuntimeError("Unexpected generation IDs exist.")

rows = [records[gid] for gid in sorted(EXPECTED_IDS)]
status_counts = Counter(x["status"] for x in rows)
print("Statuses:", dict(status_counts))

if status_counts.get("oom", 0):
    raise RuntimeError("At least one FP32 ALoDLM generation OOMed.")
if status_counts.get("error", 0):
    raise RuntimeError("At least one ALoDLM generation errored.")

GEN_DF = pd.DataFrame(rows)

assert len(GEN_DF) == 24
assert GEN_DF["generation_id"].nunique() == 24
assert (GEN_DF["generated_tokens"] > 0).all()
assert (GEN_DF["outer_steps"] > 0).all()
assert (GEN_DF["recurrent_passes"] > 0).all()

summary_rows = []
for dataset, group in GEN_DF.groupby("dataset"):
    summary_rows.append({
        "dataset": dataset,
        "generations": len(group),
        "strict_success_rate": float(group["strict_success"].mean()),
        "numeric_success_rate": float(group["numeric_success"].mean()),
        "task_success_rate": float(group["task_success"].mean()),
        "task_failures": int((~group["task_success"]).sum()),
        "mean_generated_tokens": float(group["generated_tokens"].mean()),
        "mean_outer_steps": float(group["outer_steps"].mean()),
        "mean_recurrent_passes": float(group["recurrent_passes"].mean()),
        "mean_exit_depth": float(group["mean_exit_depth"].dropna().mean())
            if group["mean_exit_depth"].notna().any() else math.nan,
        "mean_loop_token_passes": float(group["loop_token_passes"].mean()),
        "mean_wall_seconds": float(group["wall_seconds"].mean()),
        "mean_tokens_per_second": float(group["tokens_per_second"].mean()),
        "max_peak_vram_gb": float(group["peak_vram_gb"].max()),
    })

SUMMARY_DF = pd.DataFrame(summary_rows)
display(SUMMARY_DF)

print("24-generation matrix integrity: PASS")
print("Overall task success:", round(float(GEN_DF["task_success"].mean()), 4))
print("Overall task failures:", int((~GEN_DF["task_success"]).sum()))

## 13. Extract ALoDLM's own failures + progression gate

In [ ]:
FAILURE_DF = GEN_DF[~GEN_DF["task_success"]].copy()
FAILURE_DF["failure_id"] = FAILURE_DF["generation_id"]

failure_records = FAILURE_DF.to_dict("records")

print("ALoDLM own failures:", len(FAILURE_DF))
print("By dataset:", dict(Counter(FAILURE_DF["dataset"])))

display(
    FAILURE_DF[
        [
            "failure_id",
            "dataset",
            "target",
            "numeric_success",
            "generated_tokens",
            "outer_steps",
            "recurrent_passes",
            "mean_exit_depth",
            "loop_token_passes",
        ]
    ]
)

GATE = {
    "model": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "code_commit": EXPECTED_CODE_COMMIT,
    "precision": "fp32",
    "generations": int(len(GEN_DF)),
    "unique_generation_ids": int(GEN_DF["generation_id"].nunique()),
    "task_failures": int(len(FAILURE_DF)),
    "datasets_with_task_failure": int(FAILURE_DF["dataset"].nunique()),
    "unique_failed_problem_ids": int(FAILURE_DF["problem_id"].nunique()),
    "minimum_outer_steps": int(GEN_DF["outer_steps"].min()),
    "minimum_recurrent_passes": int(GEN_DF["recurrent_passes"].min()),
    "max_peak_vram_gb": float(GEN_DF["peak_vram_gb"].max()),
}

GATE["ALODLM_OWN_FAILURE_CORPUS_READY"] = bool(
    GATE["generations"] == 24
    and GATE["unique_generation_ids"] == 24
    and GATE["minimum_outer_steps"] > 0
    and GATE["minimum_recurrent_passes"] > 0
    and GATE["task_failures"] >= 6
    and GATE["datasets_with_task_failure"] >= 2
    and GATE["unique_failed_problem_ids"] >= 6
)

print(json.dumps(GATE, indent=2))

## 14. Save, hash, ZIP and download

In [ ]:
SUMMARY_PATH = OUT_DIR / "alodlm_natural_summary.csv"
FAILURES_PATH = OUT_DIR / "alodlm_own_failures.jsonl"
GATE_PATH = OUT_DIR / "ALODLM_OWN_FAILURE_GATE.json"

SUMMARY_DF.to_csv(SUMMARY_PATH, index=False)

with open(FAILURES_PATH, "w", encoding="utf-8") as f:
    for rec in failure_records:
        f.write(json.dumps(rec, ensure_ascii=False))
        f.write("\n")

GATE_PATH.write_text(
    json.dumps(GATE, indent=2),
    encoding="utf-8",
)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

artifact_paths = [
    MANIFEST_PATH,
    GENERATIONS_PATH,
    SUMMARY_PATH,
    FAILURES_PATH,
    GATE_PATH,
]

artifact_manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "run_signature": RUN_SIGNATURE,
    "embedded_payload_sha256": EMBEDDED_PAYLOAD_SHA256,
    "protocol_version": "alodlm17b-own-failures-v1",
    "files": {
        p.name: sha256_file(p)
        for p in artifact_paths
        if p.exists()
    },
}

ARTIFACT_MANIFEST_PATH = OUT_DIR / "artifact_manifest.json"
ARTIFACT_MANIFEST_PATH.write_text(
    json.dumps(artifact_manifest, indent=2),
    encoding="utf-8",
)

import shutil
ZIP_BASE = Path(
    "/content/RepairScopeX_Notebook14_ALoDLM_FP32_Own_Failures_results"
)
zip_path = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=OUT_DIR,
)

print("Created:", zip_path)
print("Size MB:", round(Path(zip_path).stat().st_size/1024**2, 3))

from google.colab import files
files.download(zip_path)

# What to send back

Send:

`RepairScopeX_Notebook14_ALoDLM_FP32_Own_Failures_results.zip`

If `ALODLM_OWN_FAILURE_CORPUS_READY = true`, Notebook 15 will use **ALoDLM's own failed responses** and test suffix self-repair with the official decoder:

- narrow suffix reopening;
- wider suffix reopening;
- global regeneration;
- task-aware correctness;
- exact preserved-prefix auditing;
- native adaptive-compute statistics.

Because the official decoder treats its prompt as observed context, the preserved response prefix can be appended to the chat prompt while only the failed suffix is regenerated.